In [0]:
%run ./supply_generator

In [0]:
%run ./distribution_center_generator

In [0]:
%run ./product_supply_generator

In [0]:
%run ./inventory_generator.py

In [0]:
%run ./purchase_order_generator

In [0]:
%run ./disruption_generator.py

In [0]:
class SimulationRunner:

    def __init__(self):

        self.supplier_generator = (
            SupplierGenerator()
        )

        self.dc_generator = (
            DistributionCenterGenerator()
        )

        self.mapper = (
            ProductSupplierMapper()
        )

        self.inventory_generator = (
            InventoryGenerator()
        )

        self.po_generator = (
            PurchaseOrderGenerator()
        )

        self.disruption_generator = (
            DisruptionGenerator()
        )

    def run(
        self,
        products_df,
        sales_df
    ):

        print(
            "\nGenerating suppliers..."
        )

        suppliers_df = (
            self.supplier_generator.generate()
        )

        print(
            f"Generated {len(suppliers_df)} suppliers"
        )

        print(
            "\nGenerating distribution centers..."
        )

        dcs_df = (
            self.dc_generator.generate()
        )

        print(
            f"Generated {len(dcs_df)} DCs"
        )

        print(
            "\nAssigning suppliers to products..."
        )

        products_df = (
            self.mapper.assign_suppliers(
                products_df,
                suppliers_df
            )
        )

        print(
            "\nGenerating inventory..."
        )

        store_inventory_df = (
            self.inventory_generator
            .generate_store_inventory(
                sales_df
            )
        )

        # Create store-to-DC mapping
        # Distribute stores evenly across DCs
        unique_stores = store_inventory_df["store_id"].unique()
        num_dcs = len(dcs_df)
        
        store_to_dc_map = {}
        for idx, store_id in enumerate(unique_stores):
            # Round-robin assignment of stores to DCs
            dc_id = (idx % num_dcs) + 1
            store_to_dc_map[store_id] = dc_id

        dc_inventory_df = (
            self.inventory_generator
            .generate_dc_inventory(
                store_inventory_df,
                store_to_dc_map
            )
        )

        print(
            "\nGenerating purchase orders..."
        )

        purchase_orders_df = (
            self.po_generator.generate(
                store_inventory_df,
                products_df,
                suppliers_df
            )
        )

        print(
            "\nGenerating disruptions..."
        )

        disruptions_df = (
            self.disruption_generator.generate(
                suppliers_df
            )
        )

        return {
            "suppliers":
                suppliers_df,

            "distribution_centers":
                dcs_df,

            "products":
                products_df,

            "store_inventory":
                store_inventory_df,

            "dc_inventory":
                dc_inventory_df,

            "purchase_orders":
                purchase_orders_df,

            "disruptions":
                disruptions_df
        }

In [0]:
if __name__ == "__main__":

    print(
        "Simulation Runner Ready"
    )